In [179]:
import pandas as pd
import numpy as np

In [180]:
pd.set_option("display.max_rows",None)
pd.set_option("display.max_columns",None)
pd.set_option("display.max_colwidth",None)

In [181]:
df = pd.read_csv("/content/cleaned.csv")

In [182]:
df.drop(columns = ['Unnamed: 0'] , inplace=True)

In [183]:
# 1

def remove_cols(df: pd.DataFrame) -> pd.DataFrame | None:
  try:
    df.drop(columns = ['address' , "balcony"], inplace=True)
    return df
  except Exception as e:
        print(f"Error categorizing address: {e}")
        return np.nan


In [184]:
df = remove_cols(df)

In [185]:
df.sample(1)

,price,area,bedrooms,bathrooms,status,neworold,furnished_status,type_of_building,price_sqft,category
1328,8.5,2650.0,5.0,4.0,Ready to Move,Resale,Semi-Furnished,Flat,32075.471698,East Mumbai


In [186]:
# 2

def change_dtype(df: pd.DataFrame) -> pd.DataFrame | None:
  try:

    df['bedrooms'] = df['bedrooms'].astype(np.int32)
    df['bathrooms'] = df['bathrooms'].astype(np.int32)
    return df

  except Exception as e:

        print(f"Error categorizing address: {e}")
        return np.nan




In [187]:
df = change_dtype(df)

In [188]:
# 3
df.drop(index=[2856],inplace=True)

In [189]:
df.shape

(5927, 10)

In [190]:
nan_status = df[df['status'].isnull()]

In [191]:
nan_status['status'] = "Under Construction"

/tmp/ipykernel_9808/1790527410.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nan_status['status'] = "Under Construction"


In [192]:
# 4
def under_cons(df: pd.DataFrame) -> pd.DataFrame | None:
  try:

    nan_status = df[df['status'].isnull()]
    nan_status['status'] = "Under Construction"
    df.update(nan_status)
    return df

  except Exception as e:

    print(f"Error categorizing address: {e}")
    return np.nan

In [193]:
df.update(nan_status)

/tmp/ipykernel_9808/606569290.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[2 3 3 ... 2 2 2]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.update(nan_status)
/tmp/ipykernel_9808/606569290.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[2 2 3 ... 2 2 2]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.update(nan_status)


In [194]:
df.sample(1)

,price,area,bedrooms,bathrooms,status,neworold,furnished_status,type_of_building,price_sqft,category
3411,1.1,1300.0,3,3,Ready to Move,Resale,Unfurnished,Flat,8461.538462,East Mumbai


In [195]:
# done

In [196]:
# File Fetaure Engg

from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer

In [197]:
ohe_cols = ["status","neworold","type_of_building","category"]

In [198]:
df['furnished_status'] = df['furnished_status'].fillna("missing")


In [199]:
mapping = {
    'Unfurnished':0,
    'Semi-Furnished':1,
    'Furnished':2,
    'missing':3
}

df['furnished_status'] = df['furnished_status'].map(mapping)

In [200]:
mapping1 = {
    0:0,
    1:1,
    2:2,
    3:np.nan
}
df['furnished_status'] = df['furnished_status'].map(mapping1)

In [201]:
df.sample(10)

,price,area,bedrooms,bathrooms,status,neworold,furnished_status,type_of_building,price_sqft,category
1984,2.00,1030.0,2,2,Ready to Move,Resale,NaN,Flat,19417.475728,East Mumbai
2,3.17,968.0,3,3,Under Construction,New Property,NaN,Flat,32747.933884,East Mumbai
2738,2.00,605.0,2,2,Under Construction,Resale,1.0,Individual House,33057.851240,West Mumbai
3088,2.70,1580.0,3,3,Under Construction,Resale,0.0,Flat,17088.607595,North Mumbai
3913,1.00,1600.0,3,4,Ready to Move,Resale,1.0,Flat,6250.000000,North Mumbai
3409,1.90,1000.0,5,5,Ready to Move,Resale,0.0,Flat,19000.000000,East Mumbai
1083,2.10,1100.0,2,2,Ready to Move,Resale,1.0,Flat,19090.909091,West Mumbai
2289,1.80,1035.0,2,2,Ready to Move,Resale,NaN,Flat,17391.304348,East Mumbai
1776,6.20,1690.0,3,4,Ready to Move,Resale,NaN,Flat,36686.390533,South Mumbai
3756,1.40,900.0,2,2,Ready to Move,Resale,NaN,Flat,15555.555556,East Mumbai


In [202]:

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

# Features used for prediction
features = [
    "price",
    "area",
    "bedrooms",
    "bathrooms"
]

# Separate known and missing values
train_df = df[df["furnished_status"].notna()].copy()
missing_df = df[df["furnished_status"].isna()].copy()

X_train = train_df[features]
y_train = train_df["furnished_status"].astype(int)

X_missing = missing_df[features]

# Model pipeline
knn_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

# Train model
knn_model.fit(X_train, y_train)

# Predict missing categories
predictions = knn_model.predict(X_missing)

# Fill missing values
df["furnished_status"] = df["furnished_status"].astype(float)

df.loc[missing_df.index, "furnished_status"] = predictions

# Check output
print(df["furnished_status"].value_counts(dropna=False))


furnished_status
0.0    2608
1.0    1933
2.0    1386
Name: count, dtype: int64


In [213]:
df['furnished_status'].isnull().sum()

np.int64(0)

In [212]:
df.sample(10)

,price,area,bedrooms,bathrooms,status,neworold,furnished_status,type_of_building,price_sqft,category
624,2.50,1209.0,2,2,Ready to Move,Resale,1.0,Flat,20678.246485,North Mumbai
2504,4.25,1150.0,3,2,Ready to Move,New Property,2.0,Flat,36956.521739,West Mumbai
58,1.53,1181.0,2,2,Under Construction,New Property,0.0,Flat,12955.122777,East Mumbai
697,1.95,1700.0,3,3,Under Construction,New Property,0.0,Flat,11470.588235,North Mumbai
3926,4.00,1200.0,2,3,Ready to Move,Resale,1.0,Flat,33333.333333,South Mumbai
2966,0.74,1250.0,2,2,Ready to Move,Resale,2.0,Flat,5920.000000,East Mumbai
4915,2.90,1880.0,4,4,Ready to Move,New Property,0.0,Individual House,15425.531915,East Mumbai
740,0.45,900.0,2,2,Ready to Move,New Property,1.0,Flat,5000.000000,North Mumbai
3802,3.80,1340.0,3,3,Ready to Move,Resale,2.0,Flat,28358.208955,West Mumbai
1023,1.40,3500.0,2,3,Ready to Move,Resale,2.0,Flat,4000.000000,North Mumbai
